In [74]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import nltk

In [75]:
data = pd.read_csv('qb_trait_scores.csv')

In [76]:
data.isna().sum()

player                        0
url                           0
rank                          0
height                        0
weight                        0
composite_rating              0
school_info                   0
city                          0
state                         0
draft_projection              9
reminds_of                   36
evaluated_date                0
analyst                      15
athletic_background           0
committed_school              0
numerical_rating              0
Is_FBS                        0
Is_SEC                        0
Is_Big_Ten                    0
Is_Big_XII                    0
Is_ACC                        0
pos_DUAL                      0
pos_PRO                       0
pos_QB                        0
star_2                        0
star_3                        0
star_4                        0
star_5                        0
Is_Top_Five_Ranked            0
Is_From_East                  0
Is_From_West                  0
weight_t

Note lot of prospects are missing certain trait categories. Specifically Football IQ, Decision Making and Leadership & Intangibles. These columns will be dropped and because of the much lower amount of missingness in the other columns, any players with missing data in the following will be dropped: Accuracy, Arm Strength, Mobility & Athleticism, Pocket Presence, Size & Physical

In [77]:
# Drop columns with lots of missing info
data = data.drop(columns=['Football IQ', 'Decision Making', 'Leadership & Intangibles'])


# Drop rows with missing values in the indicated columns
data = data.dropna(subset=['Accuracy', 'Arm Strength', 'Mobility & Athleticism', 'Pocket Presence', 'Size & Physical'])

In [78]:
data.isna().sum()

player                     0
url                        0
rank                       0
height                     0
weight                     0
composite_rating           0
school_info                0
city                       0
state                      0
draft_projection           8
reminds_of                20
evaluated_date             0
analyst                   10
athletic_background        0
committed_school           0
numerical_rating           0
Is_FBS                     0
Is_SEC                     0
Is_Big_Ten                 0
Is_Big_XII                 0
Is_ACC                     0
pos_DUAL                   0
pos_PRO                    0
pos_QB                     0
star_2                     0
star_3                     0
star_4                     0
star_5                     0
Is_Top_Five_Ranked         0
Is_From_East               0
Is_From_West               0
weight_to_height_ratio     0
bmi                        0
Accuracy                   0
Arm Strength  

### Let's look at player stats_2018_2025


ultimately, we want to look at the success players achieve at the original school they were recruited to. To do this, we will id players who have at least 2 years at the school they were recruited to and aggregated their stats together. This way we can train out models on multiple years of data for a player.

In [79]:
qb_stats = pd.read_csv(r'../Data/player_stats_2018_2025.csv', encoding='latin1')

In [80]:
qb_stats

,Rk,Player,Team,Conf,G,Cmp,Att,Cmp%,Yds,TD,TD%,Int,Int%,Y/A,AY/A,Y/C,Y/G,Rate,Awards,Year
0,1,Dwayne Haskins*,Ohio State,Big Ten,14,26.6,38.10,70.0,345.1,3.57,9.4,0.57,1.5,9.1,10.26,13.0,345.1,174.1,H-3,2018
1,2,Gardner Minshew*,Washington State,Pac-12,13,36.0,50.90,70.7,367.6,2.92,5.7,0.69,1.4,7.2,7.76,10.2,367.6,147.6,H-5,2018
2,3,Kyler Murray*,Oklahoma,Big 12,14,18.6,26.90,69.0,311.5,3.00,11.1,0.50,1.9,11.6,12.96,16.8,311.5,199.2,H-1,2018
3,4,Taylor Cornelius*,Oklahoma State,Big 12,13,22.2,37.30,59.4,306.0,2.46,6.6,1.00,2.7,8.2,8.32,13.8,306.0,144.7,NaN,2018
4,5,Tua Tagovailoa*,Alabama,SEC,15,16.3,23.70,69.0,264.4,2.87,12.1,0.40,1.7,11.2,12.83,16.2,264.4,199.4,"H-2,Maxwell,AA,Camp",2018
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3931,496,Gavin Freeman,Oklahoma State,Big 12,12,0.0,0.08,0.0,0.0,0.00,0.0,0.00,0.0,0.0,0.00,NaN,0.0,0.0,NaN,2025
3932,497,Grady Gross*,Washington,Big Ten,13,0.0,0.08,0.0,0.0,0.00,0.0,0.00,0.0,0.0,0.00,NaN,0.0,0.0,NaN,2025
3933,498,Grant Page*,Southern Mississippi,Sun Belt,11,0.0,0.09,0.0,0.0,0.00,0.0,0.00,0.0,0.0,0.00,NaN,0.0,0.0,NaN,2025
3934,499,Holden Geriner*,Texas State,Sun Belt,3,0.0,0.33,0.0,0.0,0.00,0.0,0.00,0.0,0.0,0.00,NaN,0.0,0.0,NaN,2025


In [81]:
# Take out all asterisks from player names
qb_stats['Player'] = qb_stats['Player'].str.replace('*', '', regex=False)

In [82]:
qb_stats.groupby('Player')['Team'].count().sort_values(ascending=False)

Player
Alan Bowman    7
Jacob Zeno     7
DeQuan Finn    7
Zach Gibson    6
Sam Hartman    6
              ..
AJ Maddox      1
AJ Jones       1
AJ Hill        1
AJ Duffy       1
AJ Bush        1
Name: Team, Length: 1986, dtype: int64

In [83]:
qb_stats.columns

Index(['Rk', 'Player', 'Team', 'Conf', 'G', 'Cmp', 'Att', 'Cmp%', 'Yds', 'TD',
       'TD%', 'Int', 'Int%', 'Y/A', 'AY/A', 'Y/C', 'Y/G', 'Rate', 'Awards',
       'Year'],
      dtype='str')

In [84]:
# Find players with multiple rows
multi_year_players = qb_stats.groupby(['Player','Team']).filter(lambda x: len(x) > 1)

# Average the stat columns, keep first value of categorical columns
stat_cols = ['G', 'Cmp', 'Att', 'Cmp%', 'Yds', 'TD', 'TD%', 'Int', 'Int%', 
             'Y/A', 'AY/A', 'Y/C', 'Y/G', 'Rate']
keep_cols = ['Conf']

averaged_df = (
    multi_year_players
    .groupby(['Player','Team'])
    .agg({**{col: 'mean' for col in stat_cols},
          **{col: 'first' for col in keep_cols}})
    .reset_index()
)

# Reorder columns to match original: Player, Team, Conf, stats...
averaged_df = averaged_df[['Player', 'Team', 'Conf'] + stat_cols]

In [85]:
# Check row count after averaging
averaged_df.shape

(928, 17)

In [86]:
# examine that each players is only listed once
averaged_df.groupby('Player')['Team'].count().sort_values(ascending=False)

Player
Alan Bowman          3
Chubba Purdy         3
Max Johnson          3
Athan Kaliakmanis    2
Hayden Wolff         2
                    ..
Zach Wilson          1
Zamar Wise           1
Anthony Russo        1
Zevi Eckhaus         1
Zion Turner          1
Name: Team, Length: 851, dtype: int64

We also want to save information regarding how many seasons each player played for a team. We can simply do this by saving the count from earlier and assigning it to each player

In [87]:
# save into dataframe
qb_counts = pd.DataFrame(qb_stats.groupby(['Player','Team']).size().reset_index(name='Counts'))

In [88]:
qb_counts

,Player,Team,Counts
0,A.J. Davis,UAB,1
1,A.J. Erdely,UAB,1
2,AJ Bianco,Nevada,3
3,AJ Bush,Illinois,1
4,AJ Duffy,Florida State,1
...,...,...,...
2448,Zeon Chriss,Louisiana,2
2449,Zevi Eckhaus,Washington State,2
2450,Zion Turner,Connecticut,2
2451,Zion Turner,Marshall,1


In [89]:
qb_counts[qb_counts['Player'] == 'Sam Hartman']

,Player,Team,Counts
2069,Sam Hartman,Notre Dame,1
2070,Sam Hartman,Wake Forest,5


In [91]:
merged_df = pd.merge(averaged_df, qb_counts, on=['Player', 'Team'], how='left')

In [93]:
merged_df[merged_df['Player'] == 'Sam Hartman']

,Player,Team,Conf,G,Cmp,Att,Cmp%,Yds,TD,TD%,Int,Int%,Y/A,AY/A,Y/C,Y/G,Rate,Counts
776,Sam Hartman,Wake Forest,ACC,9.6,18.66,31.78,58.44,257.08,2.036,6.2,0.79,2.44,8.08,8.236,13.84,257.08,142.0,5
